# daily_runner.py Walkthrough

`daily_runner.py` is the CLI entry point meant to be scheduled (cron/Task Scheduler/Docker) --
see `docs/RUNNING.md` for the actual scheduled commands. This notebook explains and demos its
internal pieces individually so the CLI's behavior isn't a black box.

**Note:** `main()` itself is CLI-oriented (parses `sys.argv`, calls `sys.exit()` on failure) --
this notebook calls the individual functions it's built from instead of `main()` directly.

In [ ]:
# Package is pip-installed editable, no sys.path hacking needed
from momentum_trading.daily_runner import (
    load_config, validate_config_schema,
    already_ran_today, mark_ran_today,
    check_and_handle_stop_losses, send_alert_email,
)
from momentum_trading.backtest.momentum_backtest import BacktestConfig
from momentum_trading.core.paths import data_dir
from dotenv import load_dotenv
import pandas as pd

# Loads SMTP_HOST/SMTP_PORT/SMTP_USER/SMTP_PASS/ALERT_TO_EMAIL from ../../.env into
# this process — unlike the CLI (run with the shell/Docker env already populated),
# the notebook kernel won't see .env unless we load it explicitly.
load_dotenv("../../.env")

## 1. Config loading and schema validation

`load_config()` reads `config.yaml`, validates its structure (clear errors naming the
offending portfolio/field), then builds a `BacktestConfig` per portfolio.

In [2]:
cfg_raw = load_config("../../config.yaml")
for name, spec in cfg_raw["portfolios_resolved"].items():
    print(f"{name}: tickers={spec['tickers']}, custom_weights={spec['custom_weights']}, total_value={spec['total_value']}")

portfolio1: tickers=['SPY', 'QQQ', 'XLK', 'XLF', 'XLE', 'XLY', 'XLP', 'XLU', 'GLD', 'TLT', 'BIL'], custom_weights=None, total_value=None
portfolio2: tickers=['XLF', 'XLE', 'GLD', 'TLT'], custom_weights={'XLF': 0.4, 'XLE': 0.3, 'GLD': 0.2, 'TLT': 0.1}, total_value=2500.0


In [3]:
# Deliberately broken config — shows the clear, field-specific error instead of a deep traceback
bad_config = {"portfolios": {"p1": {"tickers": ["SPY", "QQQ"], "custom_weights": {"SPY": 0.9, "QQQ": 0.9}}}}
try:
    validate_config_schema(bad_config, "example.yaml")
except ValueError as e:
    print("Caught as expected:\n", e)

Caught as expected:
 Invalid example.yaml:
  - portfolios.p1.custom_weights: weights sum to 1.8000, must be <= 1.0


## 2. Idempotency: same-day rebalance lock

Prevents a duplicate rebalance if the scheduled job fires twice (retry, manual + scheduled
overlap). `--force-rebalance` on the CLI bypasses this deliberately for testing.

In [4]:
tag = "demo_portfolio"
print("Already ran today (before):", already_ran_today(tag))
mark_ran_today(tag)
print("Already ran today (after mark):", already_ran_today(tag))

# cleanup for repeatable demo runs — lock files live in data_dir(), not the notebook's cwd
import glob, os
for f in glob.glob(str(data_dir() / f"last_run_{tag}_*.lock")):
    os.remove(f)
print("Already ran today (after cleanup):", already_ran_today(tag))

Already ran today (before): False
Already ran today (after mark): True
Already ran today (after cleanup): False


## 3. Stop-loss check: flag-only vs. auto-execute

`cfg.auto_execute_stop_loss` (default `False`) controls this. Demo with mocked positions,
always `dry_run=True` here so nothing is actually sent to a broker.

In [5]:
# Mocked position, down 15% from entry — breaches the default 12% stop_loss_pct
mock_positions = {"XLK": {"shares": 5, "avg_entry_price": 200.0}}
mock_latest_prices = {"XLK": 170.0}  # -15% from entry
demo_stoploss_log_path = str(data_dir() / "demo_stoploss_log.csv")

cfg_flag_only = BacktestConfig(stop_loss_pct=0.12, auto_execute_stop_loss=False)
flagged = check_and_handle_stop_losses(
    tickers=["XLK"], current_positions=mock_positions, latest_prices=mock_latest_prices,
    cfg=cfg_flag_only, dry_run=True, ibkr_port=7497, log_path=demo_stoploss_log_path,
)
print("Flag-only mode — flagged tickers:", flagged, "(no orders placed, email alert attempted)")

2026-07-11 03:16:18,211 | WARNING | STOP-LOSS TRIGGERED: XLK down -15.0% from entry ($200.00 -> $170.00)
2026-07-11 03:16:18,213 | WARNING | auto_execute_stop_loss=False: flagged only, no orders placed. Tickers: ['XLK']
2026-07-11 03:16:20,846 | INFO | Alert email sent: Stop-loss(es) flagged (manual review needed)


Flag-only mode — flagged tickers: ['XLK'] (no orders placed, email alert attempted)


In [6]:
cfg_auto = BacktestConfig(stop_loss_pct=0.12, auto_execute_stop_loss=True)
flagged_auto = check_and_handle_stop_losses(
    tickers=["XLK"], current_positions=mock_positions, latest_prices=mock_latest_prices,
    cfg=cfg_auto, dry_run=True, ibkr_port=7497, log_path=demo_stoploss_log_path,
)
print("Auto-execute mode (still dry_run=True) — flagged tickers:", flagged_auto)
print("Exit orders were computed and logged, but dry_run=True means nothing was sent to a broker.")
pd.read_csv(demo_stoploss_log_path).tail(2)

2026-07-11 03:16:20,861 | WARNING | STOP-LOSS TRIGGERED: XLK down -15.0% from entry ($200.00 -> $170.00)
2026-07-11 03:16:20,864 | INFO | Logged 1 order decisions to C:\Users\Denys\Downloads\test3\momentum-trading\data\demo_stoploss_log.csv (config_hash=f59aae28c3a7)
2026-07-11 03:16:20,864 | INFO | Logged 1 order decisions to C:\Users\Denys\Downloads\test3\momentum-trading\data\demo_stoploss_log.csv (config_hash=f59aae28c3a7)
2026-07-11 03:16:20,867 | INFO | DRY RUN: stop-loss exits computed but not sent to broker: ['XLK']


Auto-execute mode (still dry_run=True) — flagged tickers: ['XLK']
Exit orders were computed and logged, but dry_run=True means nothing was sent to a broker.


,timestamp,ticker,action,shares,price,reason,rank,signal_score,dry_run,config_hash,row_hash
2,2026-07-11T03:13:55.081482,XLK,SELL,5,170.0,stop-loss auto-exit,NaN,NaN,True,f59aae28c3a7,4bf557fb586fcdac
3,2026-07-11T03:16:20.864797,XLK,SELL,5,170.0,stop-loss auto-exit,NaN,NaN,True,f59aae28c3a7,15d061ecc44d8da0


## 4. Email alerting

Falls back to a visible ERROR log line if SMTP env vars aren't configured — an alert never
fails silently, it just fails loudly in a different way.

In [7]:
# With SMTP_HOST/SMTP_USER/SMTP_PASS/ALERT_TO_EMAIL unset, this logs an ERROR instead of sending:
send_alert_email("Demo Alert", "This is a demonstration alert body.")
print("\nIf you see 'SMTP env vars not fully configured — ALERT NOT SENT' above, that's the")
print("intended fallback behavior, not a bug. Set the SMTP env vars (see DEPLOYMENT.md) to test")
print("real delivery.")

2026-07-11 03:16:27,684 | INFO | Alert email sent: Demo Alert



If you see 'SMTP env vars not fully configured — ALERT NOT SENT' above, that's the
intended fallback behavior, not a bug. Set the SMTP env vars (see DEPLOYMENT.md) to test
real delivery.


## 5. Putting it together: what main() actually does

For each portfolio in `config.yaml`:
1. **If `--live`:** pull real positions + account value from IBKR (never trust local memory).
   Otherwise (dry-run): empty positions, placeholder or configured `total_value`.
2. **Always:** if there are open positions, fetch latest prices and run the stop-loss check
   (flag or auto-execute per `cfg.auto_execute_stop_loss`).
3. **If today is a scheduled rebalance day** (or `--force-rebalance`): check the idempotency
   lock, then run the full signal + sizing + order generation via `run()`, then mark the lock.
4. **On any unhandled exception:** log it, send an alert email, exit non-zero.

See `docs/RUNNING.md` for the actual scheduled commands (paper, live, single/multi-portfolio).